# Stage 3: CIEI-MERC — MELD
**Counterfactual Attention Drop (CAD) implicit edge inference**  
Novel visual stream: temporal 3-segment (begin/mid/end), OpenFace AU always present alongside CLIP or SigLIP2.  
Backbone: alternating inter/intra-modal SGConv (pure PyTorch).  
Loss: CB-Focal + EACL anchor-contrastive. 7 emotion classes.

In [1]:
import os, sys, random, json
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(42)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE      = '/mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD'
FEAT_DIR  = os.path.join(BASE, 'features')
LABELS    = os.path.join(BASE, 'labels.csv')
CKPT_DIR  = '/mnt/Work/ML/Thesis/BMVC/Hopeful/checkpoints/meld_ciei'
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Visual config ─────────────────────────────────────────────────────────────
vis_backbone  = 'siglip2'  # 'clip' | 'siglip2'
vis_pool_mode = 'attn'     # 'attn' | 'concat'

# ── Model ─────────────────────────────────────────────────────────────────────
hidden_dim   = 256
n_layers     = 4
drop         = 0.3

# ── Graph ─────────────────────────────────────────────────────────────────────
window_k     = 10
cad_k        = 3
cad_tau      = 0.5
cad_attn_dim = 128

# ── Ablation flags ────────────────────────────────────────────────────────────
use_cad   = True
use_eacl  = True
use_focal = True

# ── Loss weights ─────────────────────────────────────────────────────────────
eacl_weight  = 0.5
focal_weight = 1.0
focal_gamma  = 2.0

# ── Training ──────────────────────────────────────────────────────────────────
lr           = 1e-4
weight_decay = 1e-4
epochs       = 60
patience     = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device={device}  vis_backbone={vis_backbone}  vis_pool={vis_pool_mode}")
print(f"use_cad={use_cad}  use_eacl={use_eacl}  use_focal={use_focal}")

device=cuda  vis_backbone=siglip2  vis_pool=attn
use_cad=True  use_eacl=True  use_focal=True


## §2 Data Loading
MELD 7-class: `anger disgust fear joy neutral sadness surprise`.  
Splits via `split` column in labels.csv. Per-split feature files: `*_train.pt`, `*_dev.pt`, `*_test.pt`.  
Key format: `dia{dia_id}_utt{utt_id}` (= `clip_name` column).

In [2]:
EMOTIONS  = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
EMO2IDX   = {e: i for i, e in enumerate(EMOTIONS)}
IDX2EMO   = {i: e for i, e in enumerate(EMOTIONS)}
N_CLASSES = len(EMOTIONS)

print("Loading features (may take ~60s)...")

def load_split_feats(split):
    suf = f'_{split}.pt'
    text  = torch.load(os.path.join(FEAT_DIR, f'text_roberta_large{suf}'),          weights_only=False)
    audio = torch.load(os.path.join(FEAT_DIR, f'audio_microsoft_wavlm_large{suf}'), weights_only=False)
    if vis_backbone == 'clip':
        vis = torch.load(os.path.join(FEAT_DIR, f'video_clip_temporal{suf}'),    weights_only=False)
    else:
        vis = torch.load(os.path.join(FEAT_DIR, f'video_siglip2_temporal{suf}'), weights_only=False)
    au = torch.load(os.path.join(FEAT_DIR, f'video_openface_au{suf}'), weights_only=False)
    return text, audio, vis, au

VIS_SEM_DIM = 768 if vis_backbone == 'clip' else 1152

train_text, train_audio, train_vis, train_au = load_split_feats('train')
dev_text,   dev_audio,   dev_vis,   dev_au   = load_split_feats('dev')
test_text,  test_audio,  test_vis,  test_au  = load_split_feats('test')

# Sanity check shapes
_k = next(iter(train_text.keys()))
assert np.array(train_text[_k]).shape  == (1024,),            f"text shape: {np.array(train_text[_k]).shape}"
assert np.array(train_audio[_k]).shape == (1024,),            f"audio shape wrong"
assert np.array(train_vis[_k]).shape   == (3, VIS_SEM_DIM),   f"vis shape: {np.array(train_vis[_k]).shape}"
assert np.array(train_au[_k]).shape    == (3, 8),             f"AU shape wrong"
print(f"Features OK — text/audio (1024,) | vis (3,{VIS_SEM_DIM}) | au (3,8)")
print(f"  train keys: {len(train_text)}  dev keys: {len(dev_text)}  test keys: {len(test_text)}")

Loading features (may take ~60s)...
Features OK — text/audio (1024,) | vis (3,1152) | au (3,8)
  train keys: 9989  dev keys: 1109  test keys: 2610


In [3]:
def build_dialogues(split='train'):
    """
    Group utterances by dialogue, all 7 classes.
    Returns list of dicts: {text, audio, vis_sem, au, speakers, labels, dia_id, N}
    """
    df = pd.read_csv(LABELS)
    df = df[df['split'] == split].copy()

    # Select per-split feature dicts
    if split == 'train':
        tf, af, vf, auf = train_text, train_audio, train_vis, train_au
    elif split == 'dev':
        tf, af, vf, auf = dev_text,   dev_audio,   dev_vis,   dev_au
    else:
        tf, af, vf, auf = test_text,  test_audio,  test_vis,  test_au

    dialogues, skipped = [], 0
    for dia_id, grp in df.groupby('dia_id'):
        grp  = grp.sort_values('utt_id').reset_index(drop=True)
        keys = grp['clip_name'].tolist()

        # Drop dialogue if any utterance missing in ANY feature dict
        if any(k not in tf or k not in vf or k not in auf for k in keys):
            skipped += 1
            continue

        # Filter out utterances with unknown emotion (shouldn't happen in MELD)
        valid_mask = grp['emotion'].isin(EMOTIONS)
        if not valid_mask.all():
            grp  = grp[valid_mask].reset_index(drop=True)
            keys = grp['clip_name'].tolist()
            if len(keys) == 0:
                skipped += 1
                continue

        dialogues.append({
            'text':     torch.FloatTensor(np.stack([np.array(tf[k])  for k in keys])),  # (N,1024)
            'audio':    torch.FloatTensor(np.stack([np.array(af[k])  for k in keys])),  # (N,1024)
            'vis_sem':  torch.FloatTensor(np.stack([np.array(vf[k])  for k in keys])),  # (N,3,VIS_SEM_DIM)
            'au':       torch.FloatTensor(np.stack([np.array(auf[k]) for k in keys])),  # (N,3,8)
            'speakers': grp['speaker'].tolist(),
            'labels':   torch.LongTensor([EMO2IDX[e] for e in grp['emotion']]),
            'dia_id':   int(dia_id),
            'N':        len(keys),
        })

    total_utts = sum(d['N'] for d in dialogues)
    print(f"[{split}] {len(dialogues)} dialogues | {total_utts} utterances | {skipped} skipped")
    return dialogues

train_dias = build_dialogues('train')
dev_dias   = build_dialogues('dev')
test_dias  = build_dialogues('test')

# Class counts for CB-Focal (train only)
train_labels_flat = [l.item() for d in train_dias for l in d['labels']]
class_counts = [Counter(train_labels_flat).get(i, 1) for i in range(N_CLASSES)]
print("Class counts:", {IDX2EMO[i]: class_counts[i] for i in range(N_CLASSES)})

[train] 1037 dialogues | 9980 utterances | 1 skipped
[dev] 113 dialogues | 1099 utterances | 1 skipped
[test] 280 dialogues | 2610 utterances | 0 skipped
Class counts: {'anger': 1109, 'disgust': 271, 'fear': 268, 'joy': 1742, 'neutral': 4703, 'sadness': 683, 'surprise': 1204}


## §3 Graph Utilities
**Explicit edges**: temporal adjacency ±k, same-speaker, cross-modal triadic.  
**CAD implicit edges** (novel): influence = L2 difference of attention output with vs without utterance i in context.

In [4]:
def build_explicit_edges(N, speakers, k=10):
    """
    Node layout: text=[0..N-1]  audio=[N..2N-1]  visual=[2N..3N-1]
    Edges:
      1. Temporal adjacency (intra-modal): |i-j| <= k, bidirectional
      2. Same-speaker (intra-modal): same speaker, bidirectional
      3. Same-utterance cross-modal (t_i <-> a_i <-> v_i), bidirectional
    """
    srcs, dsts = [], []

    for off in [0, N, 2*N]:
        for i in range(N):
            for j in range(max(0, i - k), min(N, i + k + 1)):
                if i != j:
                    srcs.append(off + i); dsts.append(off + j)
        for i in range(N):
            for j in range(N):
                if i != j and speakers[i] == speakers[j]:
                    srcs.append(off + i); dsts.append(off + j)

    for i in range(N):
        ti, ai, vi = i, N + i, 2 * N + i
        for s, d in [(ti,ai),(ai,ti),(ti,vi),(vi,ti),(ai,vi),(vi,ai)]:
            srcs.append(s); dsts.append(d)

    return torch.LongTensor(srcs), torch.LongTensor(dsts)


def _get_backward_nbrs(srcs, dsts, offset, N):
    """Backward (past) intra-modal neighbours per destination node for CAD."""
    nbrs = [[] for _ in range(N)]
    for s, d in zip(srcs.tolist(), dsts.tolist()):
        ls, ld = s - offset, d - offset
        if 0 <= ls < N and 0 <= ld < N and ls < ld:
            nbrs[ld].append(ls)
    return nbrs

In [5]:
class CADModule(nn.Module):
    """
    Counterfactual Attention Drop — pseudo-causal implicit edge inference.

    Inf(i -> j) = ||f_attn(H_j | ctx_j) - f_attn(H_j | ctx_j \ {H_i})||_2  [stop-grad]
    Top-k per destination via Gumbel-Softmax straight-through.
    """
    def __init__(self, hidden_dim, attn_dim=128, top_k=3, tau=0.5):
        super().__init__()
        self.top_k = top_k
        self.tau   = tau
        self.q  = nn.Linear(hidden_dim, attn_dim, bias=False)
        self.k_ = nn.Linear(hidden_dim, attn_dim, bias=False)
        self.v  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.scale = attn_dim ** -0.5

    def _attend(self, query, keys, values):
        q    = self.q(query).unsqueeze(0)
        k    = self.k_(keys)
        attn = F.softmax(q @ k.T * self.scale, dim=-1)
        return (attn @ self.v(values)).squeeze(0)

    def forward(self, H, N, nbrs_per_node):
        by_j = defaultdict(list)
        with torch.no_grad():
            for j in range(1, N):
                ctx_idx = nbrs_per_node[j]
                if len(ctx_idx) < 2:
                    continue
                ctx_h    = H[ctx_idx]
                H_j_full = self._attend(H[j], ctx_h, ctx_h)
                for i in ctx_idx:
                    remaining = [nb for nb in ctx_idx if nb != i]
                    if not remaining:
                        score = 0.0
                    else:
                        H_j_m = self._attend(H[j], H[remaining], H[remaining])
                        score = (H_j_full - H_j_m).norm().item()
                    by_j[j].append((i, score))

        if not by_j:
            return torch.LongTensor([]), torch.LongTensor([])

        sel_srcs, sel_dsts = [], []
        for j, group in by_j.items():
            k = min(self.top_k, len(group))
            scores = torch.tensor([s for _, s in group], dtype=torch.float)
            u = torch.rand_like(scores).clamp(1e-10, 1 - 1e-10)
            perturbed = (scores + -torch.log(-torch.log(u))) / self.tau
            _, topk_idx = perturbed.topk(k)
            for idx in topk_idx.tolist():
                sel_srcs.append(group[idx][0])
                sel_dsts.append(j)

        return torch.LongTensor(sel_srcs), torch.LongTensor(sel_dsts)

In [6]:
def build_adj(srcs, dsts, n_nodes, dev):
    """Symmetric-normalised adjacency with self-loops: D^{-1/2}(A+I)D^{-1/2}"""
    sl    = torch.arange(n_nodes, device=dev)
    all_s = torch.cat([srcs.to(dev), dsts.to(dev), sl])
    all_d = torch.cat([dsts.to(dev), srcs.to(dev), sl])
    idx   = torch.stack([all_s, all_d])
    vals  = torch.ones(idx.size(1), device=dev)
    A     = torch.sparse_coo_tensor(idx, vals, (n_nodes, n_nodes)).coalesce().to_dense()
    rs    = A.sum(1).clamp(min=1e-12)
    d     = torch.diag(rs.pow(-0.5))
    return d @ A @ d


def split_intra_inter(A, N):
    """Split (3N×3N) adj into intra-modal (block-diagonal) and inter-modal."""
    A_intra = A.clone()
    for (r0, r1), (c0, c1) in [((0,N),(N,2*N)), ((0,N),(2*N,3*N)), ((N,2*N),(2*N,3*N))]:
        A_intra[r0:r1, c0:c1] = 0
        A_intra[c0:c1, r0:r1] = 0
    return A_intra, A - A_intra

## §4 Model
`TemporalVisPool` → per-modality `proj` MLPs → CAD → 4-layer alternating SGConv → late fusion → head

In [7]:
class TemporalVisPool(nn.Module):
    """
    Cat AU (3,8) to semantic visual (3,VIS_SEM_DIM) -> (3, seg_dim).
    Then either flatten (concat) or AU-weighted average (attn).
    """
    def __init__(self, sem_dim, au_dim=8, mode='attn'):
        super().__init__()
        self.mode    = mode
        self.seg_dim = sem_dim + au_dim
        if mode == 'attn':
            self.gate = nn.Linear(self.seg_dim, 1)

    @property
    def out_dim(self):
        return self.seg_dim * 3 if self.mode == 'concat' else self.seg_dim

    def forward(self, vis_sem, au):
        """vis_sem:(N,3,sem_dim)  au:(N,3,8) -> (N, out_dim)"""
        x = torch.cat([vis_sem, au], dim=-1)  # (N, 3, seg_dim)
        if self.mode == 'concat':
            return x.reshape(x.size(0), -1)
        alpha = torch.softmax(self.gate(x), dim=1)  # (N, 3, 1)
        return (alpha * x).sum(dim=1)               # (N, seg_dim)


class SGConvLayer(nn.Module):
    """D^{-1/2}(A+I)D^{-1/2} H W + H  (sym-norm GCN with residual)"""
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.W    = nn.Linear(dim, dim, bias=False)
        self.bn   = nn.BatchNorm1d(dim)
        self.act  = nn.LeakyReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, H, A):
        return self.drop(self.act(self.bn(A @ self.W(H)))) + H

In [8]:
class CIEI_MERC(nn.Module):
    def __init__(self, vis_sem_dim):
        super().__init__()
        D = hidden_dim

        self.vis_pool = TemporalVisPool(vis_sem_dim, au_dim=8, mode=vis_pool_mode)
        vis_in = self.vis_pool.out_dim

        self.proj_t = nn.Sequential(nn.Linear(1024,   D), nn.LayerNorm(D), nn.LeakyReLU(), nn.Dropout(drop))
        self.proj_a = nn.Sequential(nn.Linear(1024,   D), nn.LayerNorm(D), nn.LeakyReLU(), nn.Dropout(drop))
        self.proj_v = nn.Sequential(nn.Linear(vis_in, D), nn.LayerNorm(D), nn.LeakyReLU(), nn.Dropout(drop))

        if use_cad:
            self.cad_t = CADModule(D, cad_attn_dim, cad_k, cad_tau)
            self.cad_a = CADModule(D, cad_attn_dim, cad_k, cad_tau)
            self.cad_v = CADModule(D, cad_attn_dim, cad_k, cad_tau)

        self.gcn = nn.ModuleList([SGConvLayer(D, drop) for _ in range(n_layers)])

        self.fusion = nn.Sequential(
            nn.Linear(3 * D, D), nn.LayerNorm(D), nn.LeakyReLU(), nn.Dropout(drop)
        )
        self.head = nn.Linear(D, N_CLASSES)

    def forward_dialogue(self, dia, return_embeds=False):
        N   = dia['N']
        dev = next(self.parameters()).device

        h_v_raw = self.vis_pool(dia['vis_sem'].to(dev), dia['au'].to(dev))
        h_t = self.proj_t(dia['text'].to(dev))
        h_a = self.proj_a(dia['audio'].to(dev))
        h_v = self.proj_v(h_v_raw)

        e_s, e_d = build_explicit_edges(N, dia['speakers'], window_k)

        all_s, all_d = e_s.clone(), e_d.clone()
        if use_cad:
            for h_mod, cad_mod, off in [
                (h_t, self.cad_t, 0),
                (h_a, self.cad_a, N),
                (h_v, self.cad_v, 2 * N),
            ]:
                nbrs = _get_backward_nbrs(e_s, e_d, off, N)
                impl_s, impl_d = cad_mod(h_mod.detach(), N, nbrs)
                if impl_s.numel() > 0:
                    all_s = torch.cat([all_s, impl_s + off])
                    all_d = torch.cat([all_d, impl_d + off])

        A = build_adj(all_s, all_d, 3 * N, dev)
        A_intra, A_inter = split_intra_inter(A, N)

        H = torch.cat([h_t, h_a, h_v], dim=0)
        for l, layer in enumerate(self.gcn):
            H = layer(H, A_intra if l % 2 == 0 else A_inter)

        fused  = self.fusion(torch.cat([H[:N], H[N:2*N], H[2*N:]], dim=-1))
        logits = self.head(fused)

        return (logits, fused) if return_embeds else logits

    def forward(self, dialogues, return_embeds=False):
        return [self.forward_dialogue(d, return_embeds) for d in dialogues]

# Quick shape test
_m = CIEI_MERC(VIS_SEM_DIM).to(device)
_d = train_dias[0]
_lg, _em = _m.forward_dialogue(_d, return_embeds=True)
assert _lg.shape == (train_dias[0]['N'], N_CLASSES), f"logits shape {_lg.shape}"
assert _em.shape == (train_dias[0]['N'], hidden_dim)
print(f"Model OK — logits {_lg.shape}, embeds {_em.shape}")
print(f"Params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
del _m, _lg, _em

Model OK — logits torch.Size([14, 7]), embeds torch.Size([14, 256])
Params: 1,681,296


/tmp/ipykernel_686609/2743424282.py:8: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  A     = torch.sparse_coo_tensor(idx, vals, (n_nodes, n_nodes)).coalesce().to_dense()


## §5 Losses
`CBFocalLoss` for MELD's heavy class imbalance (disgust/fear < 3%).  
`EACLoss` anchor-contrastive for joy↔surprise, anger↔disgust confusions.

In [9]:
class CBFocalLoss(nn.Module):
    """Class-Balanced Focal Loss (Cui et al., CVPR 2019)."""
    def __init__(self, class_counts, gamma=2.0, beta=0.9999):
        super().__init__()
        self.gamma = gamma
        n   = torch.tensor(class_counts, dtype=torch.float)
        eff = (1 - beta ** n) / (1 - beta)
        w   = 1.0 / eff
        w   = w / w.sum() * len(class_counts)
        self.register_buffer('w', w)

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.w, reduction='none')
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class EACLoss(nn.Module):
    """Emotion-Anchor Contrastive Learning — learnable per-class anchor vectors."""
    def __init__(self, n_classes, embed_dim, temperature=0.07):
        super().__init__()
        self.anchors = nn.Parameter(torch.empty(n_classes, embed_dim))
        nn.init.orthogonal_(self.anchors)
        self.T = temperature

    def forward(self, embeds, labels):
        a   = F.normalize(self.anchors, dim=-1)
        e   = F.normalize(embeds,       dim=-1)
        sim = (e @ a.T) / self.T
        return F.cross_entropy(sim, labels)

## §6 Training
Use `dev_dias` for early stopping; evaluate final model on `test_dias`.

In [10]:
model      = CIEI_MERC(VIS_SEM_DIM).to(device)
focal_loss = CBFocalLoss(class_counts, gamma=focal_gamma).to(device)
eacl_loss  = EACLoss(N_CLASSES, embed_dim=hidden_dim).to(device)

optimizer  = torch.optim.AdamW(
    list(model.parameters()) + list(eacl_loss.parameters()),
    lr=lr, weight_decay=weight_decay
)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Trainable params: 1,681,296


In [11]:
def run_epoch(dias, train=True):
    model.train(train)
    total_loss, total_n = 0.0, 0
    all_pred, all_true  = [], []

    if train:
        random.shuffle(dias)

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for dia in dias:
            labels = dia['labels'].to(device)

            if train:
                optimizer.zero_grad()

            logits, embeds = model.forward_dialogue(dia, return_embeds=True)

            loss = torch.tensor(0.0, device=device)
            if use_focal:
                loss = loss + focal_weight * focal_loss(logits, labels)
            else:
                loss = loss + F.cross_entropy(logits, labels)
            if use_eacl:
                loss = loss + eacl_weight * eacl_loss(embeds, labels)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) + list(eacl_loss.parameters()), 1.0)
                optimizer.step()

            total_loss += loss.item() * dia['N']
            total_n    += dia['N']
            all_pred.extend(logits.argmax(-1).cpu().tolist())
            all_true.extend(labels.cpu().tolist())

    wf1 = f1_score(all_true, all_pred, average='weighted') * 100
    mf1 = f1_score(all_true, all_pred, average='macro')    * 100
    return total_loss / total_n, wf1, mf1

In [12]:
import time

best_dev_wf1 = 0.0
patience_ctr = 0
history      = []

print(f"{'Ep':>4} {'Tr-loss':>9} {'Tr-WF1':>8} {'Dev-WF1':>9} {'Dev-MF1':>9}  time")
print("-" * 62)

for ep in range(1, epochs + 1):
    t0 = time.time()

    tr_loss, tr_wf1, _       = run_epoch(train_dias, train=True)
    _, dev_wf1, dev_mf1      = run_epoch(dev_dias,   train=False)

    scheduler.step()
    elapsed = time.time() - t0
    history.append({'ep': ep, 'tr_loss': tr_loss, 'tr_wf1': tr_wf1,
                    'dev_wf1': dev_wf1, 'dev_mf1': dev_mf1})
    print(f"{ep:4d}  {tr_loss:9.4f}  {tr_wf1:7.2f}%  {dev_wf1:8.2f}%  {dev_mf1:8.2f}%  {elapsed:.0f}s")

    if dev_wf1 > best_dev_wf1:
        best_dev_wf1 = dev_wf1
        patience_ctr = 0
        ckpt = {
            'ep': ep, 'model': model.state_dict(),
            'eacl': eacl_loss.state_dict(),
            'dev_wf1': dev_wf1, 'dev_mf1': dev_mf1,
            'cfg': {
                'vis_backbone': vis_backbone, 'vis_pool_mode': vis_pool_mode,
                'use_cad': use_cad, 'use_eacl': use_eacl, 'use_focal': use_focal,
                'hidden_dim': hidden_dim, 'n_layers': n_layers,
            }
        }
        torch.save(ckpt, os.path.join(CKPT_DIR, 'best.pt'))
        print(f"    ↑ new best  dev WF1={dev_wf1:.2f}%  MF1={dev_mf1:.2f}%  (saved)")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stop at epoch {ep} (patience={patience})")
            break

print(f"\nBest dev WF1: {best_dev_wf1:.2f}%")

  Ep   Tr-loss   Tr-WF1   Dev-WF1   Dev-MF1  time
--------------------------------------------------------------
   1     1.2680     5.72%      2.89%      4.82%  138s
    ↑ new best  dev WF1=2.89%  MF1=4.82%  (saved)
   2     1.2184     6.35%      9.30%     10.44%  138s
    ↑ new best  dev WF1=9.30%  MF1=10.44%  (saved)
   3     1.1856     8.05%      7.44%      8.47%  137s
   4     1.1623     9.73%     13.46%     15.56%  137s
    ↑ new best  dev WF1=13.46%  MF1=15.56%  (saved)
   5     1.1295    12.72%     14.71%     17.22%  137s
    ↑ new best  dev WF1=14.71%  MF1=17.22%  (saved)
   6     1.1063    16.55%     18.82%     18.42%  137s
    ↑ new best  dev WF1=18.82%  MF1=18.42%  (saved)
   7     1.0833    19.58%     14.10%     16.21%  137s
   8     1.0571    23.18%     29.40%     26.21%  137s
    ↑ new best  dev WF1=29.40%  MF1=26.21%  (saved)
   9     1.0440    25.90%     18.33%     18.41%  137s
  10     1.0128    26.50%     27.84%     25.29%  137s
  11     0.9959    29.25%     23.80%  

## §7 Evaluation
Load best dev checkpoint → evaluate on held-out test set.

In [13]:
ckpt = torch.load(os.path.join(CKPT_DIR, 'best.pt'), weights_only=False)
model.load_state_dict(ckpt['model'])
eacl_loss.load_state_dict(ckpt['eacl'])
print(f"Loaded best model from epoch {ckpt['ep']}  dev_WF1={ckpt['dev_wf1']:.2f}%")

model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for dia in test_dias:
        logits = model.forward_dialogue(dia)
        all_pred.extend(logits.argmax(-1).cpu().tolist())
        all_true.extend(dia['labels'].tolist())

wf1 = f1_score(all_true, all_pred, average='weighted') * 100
mf1 = f1_score(all_true, all_pred, average='macro')    * 100
print(f"\nTest  WF1={wf1:.2f}%  MF1={mf1:.2f}%")
print("\nPer-class report:")
print(classification_report(all_true, all_pred, target_names=[IDX2EMO[i] for i in range(N_CLASSES)]))

Loaded best model from epoch 49  dev_WF1=42.51%

Test  WF1=43.09%  MF1=26.82%

Per-class report:
              precision    recall  f1-score   support

       anger       0.36      0.40      0.38       345
     disgust       0.22      0.06      0.09        68
        fear       0.04      0.28      0.08        50
         joy       0.36      0.30      0.33       402
     neutral       0.61      0.60      0.60      1256
     sadness       0.25      0.18      0.21       208
    surprise       0.25      0.15      0.19       281

    accuracy                           0.42      2610
   macro avg       0.30      0.28      0.27      2610
weighted avg       0.45      0.42      0.43      2610



## §8 Ablations
Key question: **how much does CAD add over explicit-only edges?**  
Also: CB-Focal vs plain CE is critical for MELD's disgust/fear (<3% of data).

In [14]:
def run_ablation(label, **overrides):
    """Train fresh model with specified overrides, return best dev WF1."""
    g = globals()
    old = {k: g[k] for k in overrides}
    g.update(overrides)

    if 'vis_backbone' in overrides:
        print(f"  [{label}]  vis_backbone change needs re-running build_dialogues — skip")
        g.update(old); return None

    seed_everything(42)
    m  = CIEI_MERC(VIS_SEM_DIM).to(device)
    fl = CBFocalLoss(class_counts, gamma=focal_gamma).to(device)
    el = EACLoss(N_CLASSES, embed_dim=hidden_dim).to(device)
    opt = torch.optim.AdamW(list(m.parameters()) + list(el.parameters()),
                            lr=lr, weight_decay=weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)

    best, pat = 0.0, 0
    for ep in range(1, epochs + 1):
        m.train(); random.shuffle(train_dias)
        for dia in train_dias:
            lbs = dia['labels'].to(device)
            opt.zero_grad()
            lg, em = m.forward_dialogue(dia, return_embeds=True)
            loss = torch.tensor(0., device=device)
            if g['use_focal']: loss = loss + focal_weight * fl(lg, lbs)
            else:              loss = loss + F.cross_entropy(lg, lbs)
            if g['use_eacl']:  loss = loss + eacl_weight * el(em, lbs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(m.parameters()) + list(el.parameters()), 1.0)
            opt.step()
        sch.step()
        m.eval(); pred, true = [], []
        with torch.no_grad():
            for dia in dev_dias:
                pred.extend(m.forward_dialogue(dia).argmax(-1).cpu().tolist())
                true.extend(dia['labels'].tolist())
        wf1 = f1_score(true, pred, average='weighted') * 100
        if wf1 > best: best, pat = wf1, 0
        else:
            pat += 1
            if pat >= patience: break

    g.update(old)
    print(f"  [{label}]  dev WF1={best:.2f}%")
    return best

print("Running ablations (each trains from scratch)\n")
results = {}
results['full']        = run_ablation('Full model (CAD+EACL+Focal)')
results['no_cad']      = run_ablation('No CAD',         use_cad=False)
results['no_eacl']     = run_ablation('No EACL',        use_eacl=False)
results['no_focal']    = run_ablation('No CB-Focal',    use_focal=False)
results['concat_pool'] = run_ablation('vis_pool=concat', vis_pool_mode='concat')
results['attn_pool']   = run_ablation('vis_pool=attn',   vis_pool_mode='attn')
print("\nAblation summary:")
for k, v in results.items():
    if v is not None: print(f"  {k:<25} dev WF1={v:.2f}%")

Running ablations (each trains from scratch)

  [Full model (CAD+EACL+Focal)]  dev WF1=40.41%
  [No CAD]  dev WF1=42.08%
  [No EACL]  dev WF1=39.17%
  [No CB-Focal]  dev WF1=45.36%
  [vis_pool=concat]  dev WF1=42.68%
  [vis_pool=attn]  dev WF1=39.68%

Ablation summary:
  full                      dev WF1=40.41%
  no_cad                    dev WF1=42.08%
  no_eacl                   dev WF1=39.17%
  no_focal                  dev WF1=45.36%
  concat_pool               dev WF1=42.68%
  attn_pool                 dev WF1=39.68%
